### LightGBM + RUS-SMOTE

Load Data Hasil Hybrid Sampling

In [ ]:
import pandas as pd
import numpy as np

print("Loading pre-saved standardized datasets...")

# Load the standardized split datasets
train_data = pd.read_csv("../dataset_samplingtrain_split_scaled_rus_smote.csv")
val_data = pd.read_csv("../dataset_splitted/val_split_scaled.csv")
test_data = pd.read_csv("../dataset_splitted/test_split_scaled.csv")

print("Loaded standardized datasets:")
print("  - train_split_scaled_rus_smote.csv:", train_data.shape)
print("  - val_split_scaled.csv:", val_data.shape)
print("  - test_split_scaled.csv:", test_data.shape)

# Separate features and target
target_col = "category"
X_train = train_data.drop(columns=[target_col]).values.astype(np.float32)
y_train = train_data[target_col].values

X_val = val_data.drop(columns=[target_col]).values.astype(np.float32)
y_val = val_data[target_col].values

X_test = test_data.drop(columns=[target_col]).values.astype(np.float32)
y_test = test_data[target_col].values

print("\nData ready for modeling:")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(f"Target classes: {sorted(np.unique(np.concatenate([y_train, y_val, y_test])))}")
print(f"Data type: {X_train.dtype}")

# For consistency with the preprocessing cell, also set df and other variables
df = pd.concat([train_data, val_data, test_data], ignore_index=True)
print(f"\nCombined dataset shape: {df.shape}")
y = df[target_col].values
print(f"Total unique classes: {len(np.unique(y))}")

### Metrik Eval LightGBM

In [ ]:
# Helper: evaluation metrics and curves for multiclass with timing
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

def evaluate_multiclass(y_true, y_proba, set_name, class_names, plot_curves, evaluation_time=None):
    """
    - y_true: integer-encoded labels, shape (n_samples,)
    - y_proba: predicted probabilities, shape (n_samples, n_classes)
    - class_names: list/array of original class names in index order
    - evaluation_time: time taken for evaluation in seconds
    """
    n_classes = len(class_names)
    y_true = np.asarray(y_true)
    y_pred = np.argmax(y_proba, axis=1)

    # Basic metrics (weighted)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Binarize for AUROC/AUPRC
    y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))

    try:
        auroc = roc_auc_score(y_true_bin, y_proba, average='weighted', multi_class='ovr')
    except Exception:
        auroc = np.nan
    try:
        auprc = average_precision_score(y_true_bin, y_proba, average='weighted')
    except Exception:
        auprc = np.nan

    # Output with timing information
    print(f"=== {set_name.upper()} METRICS ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision (weighted): {prec:.4f}")
    print(f"Recall (weighted): {rec:.4f}")
    print(f"F1 (weighted): {f1:.4f}")
    if not np.isnan(auroc):
        print(f"AUROC (OvR, weighted): {auroc:.4f}")
    else:
        print("AUROC (OvR, weighted): N/A")
    if not np.isnan(auprc):
        print(f"AUPRC (weighted): {auprc:.4f}")
    else:
        print("AUPRC (weighted): N/A")
    
    if evaluation_time is not None:
        print(f"Evaluation Time: {evaluation_time:.6f} seconds")

    # Classification report in original labels
    y_true_labels = [class_names[i] for i in y_true]
    y_pred_labels = [class_names[i] for i in y_pred]
    print("\nClassification Report:")
    print(classification_report(y_true_labels, y_pred_labels))

    if plot_curves:
        # Per-class ROC & PR
        fpr, tpr, roc_auc = {}, {}, {}
        prec_c, rec_c, pr_auc = {}, {}, {}
        
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
            prec_c[i], rec_c[i], _ = precision_recall_curve(y_true_bin[:, i], y_proba[:, i])
            pr_auc[i] = auc(rec_c[i], prec_c[i])

        # Per-class ROC & PR
        # ROC Curves - Per Class
        plt.figure(figsize=(9, 7))
        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        for i, name in enumerate(class_names):
            plt.plot(fpr[i], tpr[i], lw=1, label=f"{name} (AUC={roc_auc[i]:.3f})")
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves (Per Class) - {set_name.title()}')
        plt.legend(loc='lower right', fontsize='small')
        plt.tight_layout()
        plt.show()

        # Precision-Recall Curves - Per Class
        plt.figure(figsize=(9, 7))
        for i, name in enumerate(class_names):
            plt.plot(rec_c[i], prec_c[i], lw=1, label=f"{name} (AP={pr_auc[i]:.3f})")
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curves (Per Class) - {set_name.title()}')
        plt.legend(loc='lower left', fontsize='small')
        plt.tight_layout()
        plt.show()

    return {
        'accuracy': float(acc),
        'precision_weighted': float(prec),
        'recall_weighted': float(rec),
        'f1_weighted': float(f1),
        'auroc_weighted_ovr': None if np.isnan(auroc) else float(auroc),
        'auprc_weighted': None if np.isnan(auprc) else float(auprc),
        'evaluation_time': evaluation_time
    }

In [ ]:
import matplotlib.pyplot as plt

# Plot horizontal bar chart with custom width, value label gap, right padding, and no vertical margins
def plot_feature_importance(values, names, title, top=None, bar_height=0.6, label_gap_ratio=0.01, right_pad_ratio=0.12):
    if top is not None:
        order = np.argsort(values)[-top:]
    else:
        order = np.argsort(values)
    values_sorted = values[order]
    names_sorted = np.array(names)[order]

    fig, ax = plt.subplots(figsize=(12, 8))
    y_pos = np.arange(len(values_sorted))
    ax.barh(y_pos, values_sorted, height=bar_height, color='#5DA5DA', edgecolor='black')

    # Add a small gap between the bar end and the value label
    max_val = values_sorted.max() if values_sorted.size else 1
    gap = max_val * label_gap_ratio
    for i, v in enumerate(values_sorted):
        ax.text(v + gap, i, f"{v:.0f}", va='center', ha='left', fontsize=10)

    # Ensure the right side has extra space so labels don't hit the border
    ax.set_xlim(0, max_val * (1 + right_pad_ratio))

    # Remove vertical gaps (top/bottom) so bars touch the axes lines
    ax.set_ylim(-0.5, len(values_sorted) - 0.5)
    ax.margins(y=0)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(names_sorted)
    ax.set_xlabel('Feature importance Values')
    ax.set_ylabel('Features')
    ax.set_title(title)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    fig.tight_layout()
    plt.show()


### Train LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
import pandas as pd
import numpy as np
import json
from time import perf_counter
import matplotlib.pyplot as plt

# Use the standardized data (X_train, X_val, X_test are now scaled)
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Get the original feature names from the dataset
feature_names = train_data.drop(columns=[target_col]).columns.tolist()
print(f"Number of features: {len(feature_names)}")
print(f"Feature names: {feature_names[:24]}")

# Set feature names for the LightGBM datasets
lgb_train.set_feature_name(feature_names)
lgb_val.set_feature_name(feature_names)

# Optimal hyperparameters by optuna
params = {
    # Default
    # "data_sample_strategy": "goss", # GOSS
    # "enable_bundle": True, # EFB
    
    # Static
    "objective": "multiclass",
    "num_class": int(np.unique(y).size),
    "metric": "multi_logloss",
    "verbosity": -1,
    "seed": 42,
    
    # Hyperparameters Control
    "learning_rate": , # Fill optimal value
    "num_leaves": , # Fill optimal value
    "max_depth": , # Fill optimal value
    "min_data_in_leaf": , # Fill optimal value
    "lambda_l1": , # Fill optimal value
    "lambda_l2": , # Fill optimal value
    "feature_fraction":  # Fill optimal value
}

print("Training LightGBM model...")
# Measure training time
train_start_time = perf_counter()
model = lgb.train(
    params=params,
    train_set=lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    num_boost_round=, # Fill optimal value
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)
train_end_time = perf_counter()
training_time = train_end_time - train_start_time

print(f"\n=== TRAINING COMPLETED ===")
print(f"Training Time: {training_time:.6f} seconds")

# --- Custom Feature Importance Plots with wider bars and label gap ---
print("\n=== FEATURE IMPORTANCE ===")
imp_split = model.feature_importance(importance_type='split')
imp_gain = model.feature_importance(importance_type='gain')

# Plot Split importance
plot_feature_importance(imp_split, feature_names, 'LightGBM Feature Importance (Split)', top=None, bar_height=0.6, label_gap_ratio=0.01, right_pad_ratio=0.12)
# Plot Gain importance
plot_feature_importance(imp_gain, feature_names, 'LightGBM Feature Importance (Gain)', top=None, bar_height=0.6, label_gap_ratio=0.01, right_pad_ratio=0.12)

# --- Load label mapping for evaluation ---
label_mapping = pd.read_csv('label_mapping.csv')
class_names = label_mapping['category_name'].values

# Predictions with timing for train set
print("\nEvaluating on Training Set...")
train_pred_start_time = perf_counter()
y_train_pred_proba = model.predict(X_train)
train_pred_end_time = perf_counter()
train_evaluation_time = train_pred_end_time - train_pred_start_time

# Predictions with timing for validation set
print("\nEvaluating on Validation Set...")
val_pred_start_time = perf_counter()
y_val_pred_proba = model.predict(X_val)
val_pred_end_time = perf_counter()
val_evaluation_time = val_pred_end_time - val_pred_start_time

# Evaluate Train & Validation with timing (prints metrics + draws curves)
train_metrics = evaluate_multiclass(y_train, y_train_pred_proba, set_name="Train", 
                                  class_names=class_names, plot_curves=False, 
                                  evaluation_time=train_evaluation_time)
val_metrics = evaluate_multiclass(y_val, y_val_pred_proba, set_name="Validation", 
                                class_names=class_names, plot_curves=False, 
                                evaluation_time=val_evaluation_time)

# Also print simple validation accuracy/classification report using original labels (as before)
y_val_pred = np.argmax(y_val_pred_proba, axis=1)
y_val_labels = [class_names[idx] for idx in y_val]
y_val_pred_labels = [class_names[idx] for idx in y_val_pred]

print("\n=== VALIDATION RESULTS (Summary) ===")
print("Validation accuracy:", accuracy_score(y_val, y_val_pred))
print("\nValidation Classification Report:")
print(classification_report(y_val_labels, y_val_pred_labels))

# Save model
model_path = "saved_model_smote-rus/lightgbm_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
    
print(f"\nModel saved to: {model_path}")

# Save model info with timing information
model_info = {
    "model_type": "LightGBM",
    "num_features": X_train.shape[1],
    "feature_names": feature_names,
    "num_classes": int(np.unique(y).size),
    "validation_accuracy": float(accuracy_score(y_val, y_val_pred)),
    "best_iteration": model.best_iteration,
    "params": params,
    "class_names": class_names.tolist(),
    "timing": {
        "training_time": float(training_time),
        "train_evaluation_time": float(train_evaluation_time),
        "validation_evaluation_time": float(val_evaluation_time)
    },
    "performance_metrics": {
        "train": train_metrics,
        "validation": val_metrics
    }
}

with open("saved_model_smote-rus/lightgbm_model_info.json", 'w') as f:
    json.dump(model_info, f, indent=2)
    
print(f"Model info saved to: saved_model_smote-rus/lightgbm_model_info.json")

# Display summary table similar to the image
print("\n")
print(f"{'Model':<15} {'Dataset':<15} {'Acc (%)':<10} {'Prec (%)':<10} {'Rec (%)':<10} {'F1 (%)':<10} {'AUC (%)':<10} {'PRC (%)':<10} {'Time (s)':<12}")
print(f"{'LightGBM':<15} {'Training':<15} {train_metrics['accuracy']*100:<10.3f} {train_metrics['precision_weighted']*100:<10.3f} {train_metrics['recall_weighted']*100:<10.3f} {train_metrics['f1_weighted']*100:<10.3f} {(train_metrics['auroc_weighted_ovr']*100 if train_metrics['auroc_weighted_ovr'] else 0):<10.3f} {(train_metrics['auprc_weighted']*100 if train_metrics['auprc_weighted'] else 0):<10.3f} {training_time:<12.6f}")
print(f"{'LightGBM':<15} {'Validation':<15} {val_metrics['accuracy']*100:<10.3f} {val_metrics['precision_weighted']*100:<10.3f} {val_metrics['recall_weighted']*100:<10.3f} {val_metrics['f1_weighted']*100:<10.3f} {(val_metrics['auroc_weighted_ovr']*100 if val_metrics['auroc_weighted_ovr'] else 0):<10.3f} {(val_metrics['auprc_weighted']*100 if val_metrics['auprc_weighted'] else 0):<10.3f} {val_evaluation_time:<12.6f}")

### Test Model dengan Model yang Sudah Disimpan

In [ ]:
# Load saved model and test on test set with timing and metrics
import pickle
import json
import pandas as pd
import numpy as np
from time import perf_counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load saved model
print("Loading saved model...")
model_load_start_time = perf_counter()
model_path = "saved_model_smote-rus/lightgbm_model.pkl"
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)
model_load_end_time = perf_counter()
model_load_time = model_load_end_time - model_load_start_time

# Load model info
with open("saved_model_smote-rus/lightgbm_model_info.json", 'r') as f:
    model_info = json.load(f)

print("Model loaded successfully!")
print(f"Model loading time: {model_load_time:.6f} seconds")
print(f"Model type: {model_info['model_type']}")
print(f"Number of features: {model_info['num_features']}")
print(f"Number of classes: {model_info['num_classes']}")
print(f"Best iteration: {model_info['best_iteration']}")
print(f"Validation accuracy: {model_info['validation_accuracy']:.4f}")

# Get label encoder classes
class_names = np.array(model_info['class_names'])

# Predict on test set with timing
print("\nPredicting on test set...")
test_pred_start_time = perf_counter()
y_test_pred_proba = loaded_model.predict(X_test)
test_pred_end_time = perf_counter()
test_evaluation_time = test_pred_end_time - test_pred_start_time

y_test_pred = np.argmax(y_test_pred_proba, axis=1)

# Evaluate Test with timing (prints metrics + draws curves)
test_metrics = evaluate_multiclass(y_test, y_test_pred_proba, set_name="Test", 
                                 class_names=class_names, plot_curves=True, 
                                 evaluation_time=test_evaluation_time)

# Convert predictions back to original labels for reporting & confusion matrix
y_test_labels = [class_names[idx] for idx in y_test]
y_test_pred_labels = [class_names[idx] for idx in y_test_pred]

# Additional summary
print("\n=== FINAL TEST RESULTS ===")
print(f"Test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_test_labels, y_test_pred_labels))

# Confusion Matrix
print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test_labels, y_test_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Update model info with test results
model_info["performance_metrics"]["test"] = test_metrics
model_info["timing"]["test_evaluation_time"] = float(test_evaluation_time)
model_info["timing"]["model_load_time"] = float(model_load_time)

# Save updated model info
with open("saved_model_smote-rus/lightgbm_model_info.json", 'w') as f:
    json.dump(model_info, f, indent=2)

# Display complete performance summary table
print("\n")
print(f"{'Model':<15} {'Dataset':<15} {'Acc (%)':<10} {'Prec (%)':<10} {'Rec (%)':<10} {'F1 (%)':<10} {'AUC (%)':<10} {'PRC (%)':<10} {'Time (s)':<12}")

# Training results
train_metrics = model_info["performance_metrics"]["train"]
training_time = model_info["timing"]["training_time"]
print(f"{'LightGBM':<15} {'Training':<15} {train_metrics['accuracy']*100:<10.3f} {train_metrics['precision_weighted']*100:<10.3f} {train_metrics['recall_weighted']*100:<10.3f} {train_metrics['f1_weighted']*100:<10.3f} {(train_metrics['auroc_weighted_ovr']*100 if train_metrics['auroc_weighted_ovr'] else 0):<10.3f} {(train_metrics['auprc_weighted']*100 if train_metrics['auprc_weighted'] else 0):<10.3f} {training_time:<12.6f}")

# Validation results
val_metrics = model_info["performance_metrics"]["validation"]
val_eval_time = model_info["timing"]["validation_evaluation_time"]
print(f"{'LightGBM':<15} {'Validation':<15} {val_metrics['accuracy']*100:<10.3f} {val_metrics['precision_weighted']*100:<10.3f} {val_metrics['recall_weighted']*100:<10.3f} {val_metrics['f1_weighted']*100:<10.3f} {(val_metrics['auroc_weighted_ovr']*100 if val_metrics['auroc_weighted_ovr'] else 0):<10.3f} {(val_metrics['auprc_weighted']*100 if val_metrics['auprc_weighted'] else 0):<10.3f} {val_eval_time:<12.6f}")

# Test results
test_eval_time = model_info["timing"]["test_evaluation_time"]
print(f"{'LightGBM':<15} {'Testing':<15} {test_metrics['accuracy']*100:<10.3f} {test_metrics['precision_weighted']*100:<10.3f} {test_metrics['recall_weighted']*100:<10.3f} {test_metrics['f1_weighted']*100:<10.3f} {(test_metrics['auroc_weighted_ovr']*100 if test_metrics['auroc_weighted_ovr'] else 0):<10.3f} {(test_metrics['auprc_weighted']*100 if test_metrics['auprc_weighted'] else 0):<10.3f} {test_eval_time:<12.6f}")